# Ten-target apo density with a Gaussian-splat decoder

Overfitting on **10 CrossDocked test targets**
(2, 3, 4, 5, 7, 8, 9, 10, 11, 12) with **apo density**
and a **Gaussian-splat decoder**.

> **This run uses apo density: the ligand density is masked out.** Each input is
> the measured 2Fo-Fc map of the complex with the ligand's own density removed
> using FuncBind's ligand occupancy field, and the vacated voxels refilled with
> the crop's bulk-solvent level. The encoder sees each pocket as if nothing were
> bound. Across all ten targets, at most 0.0707 of the voxels within 2 Å
> of a ligand atom still exceed 1.5σ. No separately measured apo crystal
> structure is involved.

Two things differ from the earlier single-target run, and both remove a way that
result could have been explained by memorization:

1. **One density adapter across all ten targets**
   (1,118,720 parameters, 3,000 steps).
   A single-target adapter sees a constant density feature and can reach its one
   reference code by memorizing a constant residual. An adapter shared by ten
   targets cannot.
2. **One Gaussian-splat decoder across all ten targets**
   (107,088 parameters, best step
   19,250). There is no pretrained Gaussian decoder, so it is
   fitted here on the frozen encoder's codes for these ten targets — replacing
   the INR decoder used in the earlier reports.


In [ ]:
import json
from pathlib import Path

import numpy as np

search_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    root for root in search_roots
    if (root / 'exps' / 'density_fusion').exists()
)
RESULT_DIR = PROJECT_ROOT / 'exps/density_fusion/apo_gaussian_10targets_20260730'
with (RESULT_DIR / 'result.json').open() as handle:
    report = json.load(handle)
report['experiment'], report['targets']


## Result across the ten targets

| Method | mIoU (mean ± sd) | Atom F1 | Density MSE | Targets improved |
|---|---:|---:|---:|---:|
| Original FuncBind | 0.227 ± 0.083 | 0.506 ± 0.194 | 1.058e-04 | — |
| Apo-density adapter | 0.253 ± 0.064 | 0.552 ± 0.129 | 9.590e-05 | 6/10 |

Every target is sampled with 8 chains that share
one initial latent and a restored sampler RNG state between the two arms, so the
comparison is paired per target. "Targets improved" counts how many of the ten
had a lower latent MSE with the apo-density adapter than without it.

![Per-target results](figures/apo_gaussian_per_target.png)


## Per-target detail

| Target | Original FuncBind mIoU / F1 | Apo-density adapter mIoU / F1 |
|---|---|---|
| 2 | 0.251 / 0.727 | 0.345 / 0.784 |
| 3 | 0.184 / 0.286 | 0.251 / 0.629 |
| 4 | 0.264 / 0.500 | 0.227 / 0.581 |
| 5 | 0.096 / 0.178 | 0.127 / 0.400 |
| 7 | 0.230 / 0.483 | 0.250 / 0.596 |
| 8 | 0.200 / 0.471 | 0.321 / 0.429 |
| 9 | 0.257 / 0.739 | 0.315 / 0.636 |
| 10 | 0.096 / 0.302 | 0.172 / 0.320 |
| 11 | 0.364 / 0.745 | 0.282 / 0.615 |
| 12 | 0.326 / 0.632 | 0.239 / 0.529 |

![Paired deltas](figures/apo_gaussian_paired_deltas.png)


## The apo density inputs

![Masked density per target](figures/apo_gaussian_masks.png)

| Target | Density at ligand atoms (before → after) | Voxels ≥1.5σ within 2 Å (before → after) |
|---|---:|---:|
| 2 | 1.14 → -0.05 | 0.056 → 0.0000 |
| 3 | 0.31 → 0.06 | 0.000 → 0.0000 |
| 4 | 2.20 → 0.11 | 0.234 → 0.0485 |
| 5 | 1.93 → -0.14 | 0.162 → 0.0026 |
| 7 | 1.60 → 0.16 | 0.105 → 0.0033 |
| 8 | 1.09 → -0.04 | 0.215 → 0.0707 |
| 9 | 0.48 → 0.10 | 0.026 → 0.0029 |
| 10 | 0.64 → -0.04 | 0.052 → 0.0034 |
| 11 | 2.54 → -0.01 | 0.230 → 0.0053 |
| 12 | 1.09 → -0.46 | 0.070 → 0.0067 |

Two caveats this table makes visible, both of which qualify how much any single
target is worth:

- **How much ligand density there was to remove varies widely.** Where the
  "before" value is low the ligand sits in weak density — poorly ordered in the
  crystal, or imperfectly aligned to the pocket crop — so masking changes little
  and that target is weak evidence in either direction.
- **The leakage column is an upper bound, not pure ligand density.** It counts
  every voxel above 1.5σ within 2 Å of a ligand atom, which includes
  neighbouring *receptor* density that is deliberately kept. It therefore reads
  high in tight pockets even when the ligand was fully erased.


## 3D occupancy comparison

One row per target. The first panel shows the **apo density actually fed to the
encoder** — points at ≥1.5 local σ, with the reference ligand atoms drawn on top
so the emptied ligand site is visible. The remaining panels use the same element
colours at an occupancy threshold of 0.1, with the reference atoms repeated in
every panel as the common ground truth.

Both generated arms are decoded by the shared Gaussian-splat decoder.

![3D occupancy comparison 1](figures/apo_gaussian_occupancy_3d_1.png)

![3D occupancy comparison 2](figures/apo_gaussian_occupancy_3d_2.png)


## The two joint fits

The adapter and the Gaussian decoder each serve all ten targets. The decoder
panel shows the mean validation loss and the worst single target, so a decoder
that fits nine targets and fails one is visible rather than averaged away.

![Joint fits](figures/apo_gaussian_fits.png)


## The existing FuncBind decoder versus the Gaussian-splat decoder

Neither table below is from this run's ten CrossDocked targets. Both come from
the same-target decoder ablation on **MCPP target 0** (`1bm2-CP.pdb`),
which is the only place the pretrained INR decoder and a Gaussian-splat decoder
were measured on identical inputs. They are reported here because this notebook
replaces the INR decoder, and that substitution needs its own reference point.

### Reconstructing one reference code (full 128³ grid)

| Decoder | Parameters | Density MSE | Density MAE | mIoU | Atom P / R / F1 | Coord RMSD (Å) | Element acc. |
|---|---:|---:|---:|---:|---:|---:|---:|
| Original FuncBind INR · posterior sample | 62,258,696 | 5.159e-07 | 1.463e-04 | 0.953 | 1.000 / 1.000 / 1.000 | 0.258 | 1.000 |
| Original FuncBind INR · posterior mean | 62,258,696 | 3.367e-07 | 4.930e-05 | 0.959 | 1.000 / 1.000 / 1.000 | 0.252 | 1.000 |
| Gaussian splat · single-target overfit | 107,088 | 5.752e-06 | 6.942e-04 | 0.844 | 1.000 / 1.000 / 1.000 | 0.262 | 1.000 |

The Gaussian decoder is **581× smaller** — 107,088
parameters against 62,258,696. The encoder
(58,921,152 parameters) is shared and frozen in both cases, so the
whole pipeline goes from 121.2M to
59.0M parameters — the decoder falls from
51% of the pipeline to
0.18%.

What that costs is density fidelity: the INR decoder reaches
11–17× lower MSE and about
0.11 higher mIoU. What it does not cost is atom placement — atom
precision, recall, F1, and element accuracy all saturate at 1.000 for every row,
so they do not separate the decoders at all, and coordinate RMSD spans only
0.011 Å across the three. The Gaussian decoder's deficit
is in the density field it paints, not in where it puts the atoms.

### What each Gaussian decoder fit cost

| Gaussian decoder fit | Targets | Parameters | Steps | Best step | Best val loss | Worst single target |
|---|---:|---:|---:|---:|---:|---:|
| Single target (MCPP target 0) | 1 | 107,088 | 5,000 | 4,300 | 6.96e-06 | — |
| Joint, this run (10 CrossDocked targets) | 10 | 107,088 | 20,000 | 19,250 | 7.61e-06 | 1.28e-05 |

Ten codes cost the same 107,088-parameter decoder almost
nothing: 7.61e-06 jointly against
6.96e-06 on one target, and even its worst single
target (1.28e-05) stays within a factor of two of the single-target fit. The
single-target run is the easier problem and is the reference, not a baseline to
beat.


## How to read this

The single-target run had an escape hatch: with one target the density feature
is a constant, so the adapter could reach its one reference code by learning a
constant residual, and a map with the ligand erased scored just as well as the
map with it. That is why the earlier result was read as memorization.

Sharing one adapter across 10 targets closes that
particular hatch. A constant residual cannot serve ten different reference
codes, so any per-target effect has to come from the only per-target input the
adapter receives — the density feature.

**But this is still an in-sample result, and one specific loophole remains
open.** The adapter was fitted on these same ten targets, and each target's
density feature is effectively a unique fingerprint. A shared adapter can
therefore still memorize *per-target* residuals by keying off that fingerprint —
a lookup table rather than a constant. That is memorization too, and nothing in
this notebook can distinguish it from the adapter genuinely reading pocket
density.

Only a held-out target — one whose density the adapter never saw during
fitting — separates the two. That experiment is the necessary next step, and it
is not what was run here.

One thing this run does establish on its own terms: the Gaussian-splat decoder
holds up. A single 107,088-parameter decoder fitted
across all ten codes reached a validation loss of
7.61e-06, against 6.96e-06 for the same
decoder overfit on a *single* target, and it does so with
0.17% of the original
INR decoder's parameters. Ten codes cost it almost nothing.

## Limitations

- The apo map is a masked holo map, not a measured apo structure: solvent reorganization and side-chain relaxation on ligand release are not modeled, and receptor density inside the ligand envelope is erased with the ligand.
- The Gaussian decoder is fitted on these same ten targets' codes, so decoded quality is in-sample and is not a held-out decoder result.
- Chain selection per target and arm is an oracle on latent MSE.
- Ten targets is a small sample; per-target pairing is reported so the spread is visible rather than hidden in a mean.
- The shared adapter was fitted on these same ten targets, so a per-target density feature can act as a target fingerprint and be memorized. Held-out targets are required to rule this out.
